# Export & Vectorization — Monitor do Fogo (Brasil)

Pipeline de 4 etapas para exportar, mosaicar, vetorizar e publicar os mapas mensais
de area queimada do Monitor do Fogo no Brasil.

**ImageCollection origem:** `projects/mapbiomas-public/assets/brazil/fire/monitor/mapbiomas_fire_monthly_burned_v1`  
**Asset GEE destino:** `projects/mapbiomas-workspace/FOGO/MONITORAMENTO/fire_monitor_v1_monthly_burned_brazil_vectors/`  
**Bucket GCS:** `mapbiomas-fire/monitor/`

---

## Fluxo

```
GEE ImageCollection
       |
       v  [1] Export (export.py)
tiles .tif no GCS  (monitor/monthly_images/temp/)
       |
       v  [2] Mosaic (mosaic.py)
mosaico COG no GCS  (monitor/monthly_images/monthly_burned/)
       |
       v  [3] Vectorize (vectorize.py)
shapefile no GCS  (monitor/monthly_vectors/monthly_burned/)
       |
       v  [4] Upload GEE (vectorize.py)
FeatureCollection no GEE
```

## Como usar

1. Execute as celulas 2-4 em ordem (instalar, clonar, autenticar).
2. Celula 5: abre a UI. Clique em **Sincronizar** para carregar o estado atual.
3. Selecione os meses pendentes nos checkboxes.
4. Execute as celulas 6-9 em ordem, uma de cada vez:
   - Celula 6: Export (GEE -> GCS tiles) — **lento**, aguarde as tasks.
   - Depois de finalizar, clique Sincronizar na UI para atualizar a grid.
   - Celula 7: Mosaic — **paralelo** (usa todos os CPUs).
   - Celula 8: Vectorize — **paralelo**.
   - Celula 9: Upload GEE — sequencial.


In [ ]:
!apt-get update -q
!apt-get install -y gdal-bin python3-gdal
!pip install --quiet gcsfs earthengine-api geopandas rasterio psutil ipywidgets

print('Dependencias instaladas.')

In [ ]:
!rm -rf /content/brazil-fire
!git clone --branch direct_link_to_download https://github.com/mapbiomas/brazil-fire.git /content/brazil-fire

import sys
sys.path.append('/content/brazil-fire/mapbiomas_fire_monitor/version_01')

print('Repositorio clonado e path configurado.')

In [ ]:
# ╔══════════════════════════════════════════╗
# ║      CONFIGURACAO — edite aqui         ║
# ╚══════════════════════════════════════════╝
GEE_PROJECT  = "ee-ipam"
BUCKET_GCS   = "mapbiomas-fire"
EXPORT_FLAG  = "WS_fire_monitor-"

from google.colab import auth
auth.authenticate_user()

import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

import export_and_vectorization.state as _state
_state.GEE_PROJECT = GEE_PROJECT
_state.BUCKET = BUCKET_GCS

import export_and_vectorization.export as _export
_export.EXPORT_FLAG = EXPORT_FLAG

print('Autenticacao concluida. Projeto GEE:', GEE_PROJECT, '| Bucket:', BUCKET_GCS)

In [ ]:
from export_and_vectorization.ui import run_ui

ui = run_ui()

In [ ]:
from export_and_vectorization.export import export_selected

export_selected(ui, logger=ui._log)

In [ ]:
from export_and_vectorization.mosaic import mosaic_selected

mosaic_selected(ui, logger=ui._log)

In [ ]:
from export_and_vectorization.vectorize import vectorize_selected

vectorize_selected(ui, logger=ui._log)

In [ ]:
from export_and_vectorization.vectorize import gee_upload_selected

gee_upload_selected(ui, logger=ui._log)